<a href="https://colab.research.google.com/github/MDunn83/Doc-Intelligence-Tool/blob/main/SHALL_Critique_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U "langchain-core<0.3.0" "langchain<0.3.0" langchain-community langchain-google-genai langchain_chroma
!pip install pypdf
!pip install flashrank
print('\n✅ Installation complete. Please go to Runtime -> Restart Session before proceeding.')

In [ ]:
# Securely fetch the API key from Colab Secrets
import os
from google.colab import userdata

try:
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    print('✅ Google API Key loaded from Secrets.')
except Exception as e:
    print('❌ Error: Please add GOOGLE_API_KEY to the Secrets (🔑) tab in the left sidebar.')

In [ ]:
import os

#Identify files to load
pdf_files = [
    "MIL-STD-882E.pdf",
    "DOD SysEng Guidebook.pdf",
    "Scrum-Guide-US-2020.pdf",
    "DOD_DevSecOps Fundamentals.pdf"
]

print("--- Checking for Standards Documents ---")
#Ensures docs are available to load
for pdf in pdf_files:
    if os.path.exists(pdf):
        print(f"✅ Found: {pdf}")
    else:
        print(f"❌ Missing: {pdf} (Please upload this to the root folder)")

# Also check the requirements file used in the audit
req_path = "/content/PayCore360_Requirements.pdf"
if os.path.exists(req_path):
    print(f"✅ Found: {req_path}")
else:
    print(f"❌ Missing: {req_path}")

In [3]:
#Initialize code
import os
import sys
sys.path.append('../..')
import shutil  #ensures old data base is cleared
if os.path.exists('./docs/chroma'):
    shutil.rmtree('./docs/chroma')

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

In [ ]:
from langchain_community.document_loaders import PyPDFLoader #call PDF Loader

#identify files to load
pdf_files = [
    "MIL-STD-882E.pdf",
    "DOD SysEng Guidebook.pdf",
    "Scrum-Guide-US-2020.pdf",
    "DOD_DevSecOps Fundamentals.pdf"
]

#load each file and ID if the load passed or failed and ID number of pages
pages = []
for pdf in pdf_files:
    try:
        loader = PyPDFLoader(pdf)
        pages.extend(loader.load())
        print(f"✅ Loaded: {pdf}")
    except Exception as e:
        print(f"❌ Failed: {pdf} — {e}")

print(f"\nTotal pages loaded: {len(pages)}")

In [ ]:
import shutil
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.chat_message_histories import ChatMessageHistory

# Direct import to bypass any broken 'langchain_core.memory' mapping
try:
    from langchain.memory.buffer import ConversationBufferMemory
except ImportError:
    from langchain.memory import ConversationBufferMemory

# 1. Define the directory variable first
persist_directory = './chroma_db_v4'

# 2. Clear the database folder to prevent 'locked file' errors
if os.path.exists(persist_directory):
   shutil.rmtree(persist_directory)

# 3. Split the documents
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    length_function=len
)
pages_to_split = pages
docs = text_splitter.split_documents(pages_to_split)
print(f"Chunks created: {len(docs)}")

# 4. Initialize Embeddings
embedding = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

In [ ]:
# 5. Create the vector database
vectordb = Chroma.from_documents(
    documents=docs,
    embedding=embedding,
    persist_directory=persist_directory
)

print(f"Final VectorDB count: {vectordb._collection.count()}")

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.memory import ConversationBufferMemory
from google.colab import userdata

# 6. Initialize LLM and Memory
# Using the exact version string 'gemini-2.0-flash-001' which appears in list_models() output
llm = ChatGoogleGenerativeAI(
    model="gemini-flash-latest", # Changed to a recommended available model
    google_api_key=userdata.get('GOOGLE_API_KEY'),
    temperature=0
)

memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True
)

print(f"✅ LLM initialized with: {llm.model}")

In [ ]:
import google.generativeai as genai
from google.colab import userdata

# Configure API and ensure your model is avaialble
api_key = userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=api_key)

print("--- Available Models for your API Key ---")
try:
    for m in genai.list_models():
        if 'embedContent' in m.supported_generation_methods:
            print(f"✅ Embedding Model: {m.name}")
        elif 'generateContent' in m.supported_generation_methods:
            print(f"🔹 Chat/Generation Model: {m.name}")
except Exception as e:
    print(f"❌ Error listing models: {e}")

In [ ]:
# We are DEFINING the tool here called "get_document_answer" using variables "user_query", "pipeline" and "history"
def get_document_answer(user_query, pipeline, history):
    """
    Executes a query using the modern .invoke() method and handles history.
    """
    # We pass BOTH the question and an empty list for chat_history
    # Using .invoke() instead of () fixes Deprecation Warnings
    result = pipeline.invoke({
        "question": user_query,
        "chat_history": history #Use the history passed
    })

    history.append((user_query, result['answer']))

    print(f"ANALYST RESPONSE:\n{result['answer']}\n") #llm response (no meta data yet)

    print("CITATIONS:") #prints citation for information returned including doc name, and page number
    seen_sources = set()
    for doc in result["source_documents"]:
        source_name = doc.metadata.get('source', 'Unknown Document')
        page_num = doc.metadata.get('page', 'N/A')
        citation = f"Ref: {source_name} (Page {page_num})"

        if citation not in seen_sources: #prevents same page number from being printed multiple times
            print(f"  - {citation}")
            seen_sources.add(citation)
    print("-" * 40)       #separator

    return history  # 3. Hand back the updated list

from langchain.chains.query_constructor.base import AttributeInfo  #calls schema tool for data to define source and page as metadata
from langchain.retrievers.self_query.base import SelfQueryRetriever #calls self query tool

metadata_field_info = [
    AttributeInfo(
        name="source",
        description="The name of the PDF document (e.g., MIL-STD-882E.pdf)",
        type="string",
    ),
    AttributeInfo(
        name="page",
        description="The specific page number in the document",
        type="integer",
    ),
]

document_content_description = "Technical guidebooks and safety standards"

# This allows the llm to search only one document if a user specifies the document in the query
self_query_retriever = SelfQueryRetriever.from_llm(
    llm,
    vectordb,
    document_content_description,
    metadata_field_info,
    search_kwargs={"k": 10}, # defines number of candidate answers to select
    verbose=True # Set this to True to see the 'Filter' in action
)

# 1. Initialize the ranker WITH the model name (Crucial for 2026)
from flashrank import Ranker  #calls ranker tool
ranker = Ranker(model_name="ms-marco-MiniLM-L-12-v2") #initialize cross-encoder model to look at meanging between query and result

# 2. This is the correct, updated path for 2026
from langchain_community.document_compressors.flashrank_rerank import FlashrankRerank #integration for the ranker tool initialized above
from langchain.retrievers import ContextualCompressionRetriever #invokes tool to to wrap the retriever and ranker

# 4. Use a Re-ranker to sort them by actual relevance (in the current case; 3)
compressor = FlashrankRerank(model="ms-marco-MiniLM-L-12-v2", top_n=3)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=self_query_retriever  # We use the self_query_retriever as the engine inside the envelope
)

from langchain.chains import ConversationalRetrievalChain #invoke tool to connect the LLM, the Memory, and the Retriever.

chat_history = []

qa_chain = ConversationalRetrievalChain.from_llm(
    llm,
    retriever=compression_retriever,
    return_source_documents=True,
)


In [ ]:
#example queries - Commented out to skip execution
# chat_history = get_document_answer("What does the Scrum guide say?", qa_chain, chat_history)
# chat_history = get_document_answer("tell me about Agile?", qa_chain, chat_history)
# chat_history = get_document_answer("what are the consequences of poor requirements?", qa_chain, chat_history)

In [10]:
#This cell creates a temporary data base for the document under review
import re
import os
import time
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader

def precision_requirement_audit(input_pdf, standards_pipeline):
    """
    Surgically extracts SHALL statements, stores them in a temp DB,
    and audits each one exactly once against the standards.
    """
    # 1. Load and Flatten
    loader = PyPDFLoader(input_pdf)
    pages = loader.load()
    full_text = " ".join([p.page_content for p in pages]).replace('\n', ' ')

    # 2. Extract Sentences containing "SHALL"
    found_sentences = re.findall(r'[^.!?]*\bSHALL\b[^.!?]*[.!?]', full_text, flags=re.IGNORECASE)

    # 3. NEW: Filter out known non-requirement "SHALL" statements (e.g., section titles, introductory remarks)
    # This filter aims to be more general by identifying phrases that introduce content rather than define system behavior.
    filtered_sentences = []
    for s in found_sentences:
        lower_s = s.strip().lower()

        # Heuristic: Filter out sentences containing "SHALL" that also refer to document structure or lists
        # e.g., "The following table defines...", "This section shall outline...", "Below are the requirements..."
        if re.search(r'\bshall\b.*?\b(the following|this section|below|as follows)\b', lower_s):
            print(f"Filtering out potential header (structure reference): '{s.strip()}'") # Debug print
            continue

        filtered_sentences.append(s)

    # 4. Ensure Uniqueness (now from filtered_sentences)
    unique_requirements = list(dict.fromkeys([s.strip() for s in filtered_sentences]))

    # 5. Create atomic documents for the Temporary VectorDB
    audit_docs = [Document(page_content=req, metadata={"req_id": i})
                  for i, req in enumerate(unique_requirements)]

    # 6. Build Temporary VectorDB
    temp_db = Chroma.from_documents(
        documents=audit_docs,
        embedding = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001"),
        collection_name="temp_audit_session"
    )
    print(f"✅ Audit Initialized. {len(unique_requirements)} unique requirements identified.")

    # 7. Initialize local history for this audit session
    audit_history = []

    for i, req_text in enumerate(unique_requirements, 1):
        audit_query = (f"You are a Quality Auditor. Here is a requirement: '{req_text}'\n\n"
                      "Compare this to the 'Best Practices' in your standards database.")
        print(f"\n--- [Auditing Requirement {i}/{len(unique_requirements)}] ---")

        # Passing 'audit_history' as the third required argument
        audit_history = get_document_answer(audit_query, standards_pipeline, audit_history)

    # 8. Final Cleanup
    temp_db.delete_collection()
    print("\n✨ Audit Complete. Temporary database purged.")

In [ ]:
import os

# Path to your uploaded mock requirements; update as required
requirements_pdf = "/content/PayCore360_Requirements.pdf"

# Verify and execute
if os.path.exists(requirements_pdf):
    dry_run_requirement_audit(requirements_pdf)
else:
    print(f"❌ File '{requirements_pdf}' not found.")

In [13]:
# FINAL EXECUTION: Run the AI-powered audit.
# Ensure your API key is in Secrets and the first cell has been run.

requirements_pdf = "/content/PayCore360_Requirements.pdf"

# To run the REAL audit:
precision_requirement_audit(requirements_pdf, qa_chain)

# To run a DRY RUN (no API costs):
# dry_run_requirement_audit(requirements_pdf)

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Filtering out potential header (structure reference): 'System SHALL Requirements The following table defines the functional requirements for PayCore 360.'
✅ Audit Initialized. 10 unique requirements identified.

--- [Auditing Requirement 1/10] ---


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


ANALYST RESPONSE:
As a Quality Auditor, I have evaluated **REQ-001** against the "Best Practices" and "Design Considerations" outlined in the provided standards documentation (specifically Section 5).

### **Requirement Evaluation: REQ-001**
> *"The system SHALL process payroll calculations for all employees and generate pay stubs within 60 seconds of initiating a payroll run, for a workforce of up to 10,000 employees."*

---

### **Comparison to Standards & Best Practices**

#### **1. Performance and Scalability (General SE Principles)**
*   **Observation:** The requirement is well-defined in terms of performance metrics (60 seconds) and capacity (10,000 employees). 
*   **Best Practice Alignment:** The documentation emphasizes that PMs and Systems Engineers should address performance constraints early to "achieve better mission performance and to preclude a stovepipe view during design." REQ-001 successfully quantifies these constraints.

#### **2. Accessibility (Section 5.1 - Sectio

In [ ]:
# Use this cell to LOAD the database
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings

persist_directory = './chroma_db_v4'
embedding = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

# Load the database from disk
vectordb = Chroma(
    persist_directory=persist_directory,
    embedding_function=embedding
)

print(f"✅ VectorDB reloaded from disk. Total chunks: {vectordb._collection.count()}")

✅ VectorDB reloaded from disk. Total chunks: 1249
